# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Sujan-lab-cell/flyrank-ml-internship/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 0. Setup (Colab or local)
On Colab this clones the repo and sets working directory to the repo root. Locally it locates the repo root.

In [1]:
import os, sys, subprocess

IN_COLAB = 'google.colab' in sys.modules
REPO_URL = 'https://github.com/Sujan-lab-cell/flyrank-ml-internship'
REPO_DIR = 'flyrank-ml-internship'

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.isdir('data/raw') and os.getcwd() != os.path.dirname(os.getcwd()):
        os.chdir('..')

print('Working dir:', os.getcwd())
assert os.path.exists('data/raw/content_refresh_anonymized.csv'), 'starter CSV not found — are you at the repo root?'
print('Starter data found. You are ready.')

Working dir: /content/flyrank-ml-internship
Starter data found. You are ready.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

**ML Task Framing:** **Supervised priority scoring / ranking** using a binary decline outcome as a starter proxy.

**Why this task type:** The content team needs a prioritized review queue rather than simply a list of pages labeled as declining or not declining. A supervised model can learn from page-level signals and produce a score for each page. These scores can then be sorted to identify which pages should be investigated first. This directly supports the Lane 2 decision: **which existing pages should receive human content review first?**

In [2]:
# Code check: Inspect target label binary distribution and basic ranking frame
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

print("=== TARGET LABEL DISTRIBUTION ===")
print(f"Total Pages: {len(df):,}")
print(f"Positive Class (Declining = 1): {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean():.1%})")
print(f"Negative Class (Stable/Up/Other = 0): {(df['is_declining_label'] == 0).sum():,} ({(df['is_declining_label'] == 0).mean():.1%})")

=== TARGET LABEL DISTRIBUTION ===
Total Pages: 30,000
Positive Class (Declining = 1): 16,262 (54.2%)
Negative Class (Stable/Up/Other = 0): 13,738 (45.8%)


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

### Target Definition

- **Proxy Label (Starter Dataset):** `is_declining_label = (trend_direction == 'down')`. This converts the starter dataset's observed `trend_direction` into a binary outcome:
  - `1` = page is labeled `down`
  - `0` = page is labeled `stable`, `up`, `new`, or `flat`

- **Future Observed Outcome (Capstone Direction):** A stronger final target should be based on a measurable outcome in a later time window, using only information available before that outcome occurs. The exact final target will be defined in the later data-contract/modeling stage.

- **Target Leakage Safeguard:** Because `is_declining_label` is derived directly from `trend_direction`, both `trend_direction` and `trend_pct` must be excluded from the feature matrix `X`. Identifiers such as `client_id` and `content_id` are also excluded from model features.

In [3]:
# Code check: Target leakage audit — verifying excluded target derivations
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

all_cols = set(df.columns)
target_cols = {'trend_direction', 'trend_pct', 'is_declining_label'}
id_cols = {'content_id', 'client_id'}
feature_cols = [c for c in df.columns if c not in target_cols and c not in id_cols]

print(f"Total Columns: {len(all_cols)}")
print(f"Target & Derivations (EXCLUDED): {target_cols}")
print(f"Identifiers (CONTEXT ONLY): {id_cols}")
print(f"Potential Features Count: {len(feature_cols)}")

Total Columns: 44
Target & Derivations (EXCLUDED): {'is_declining_label', 'trend_pct', 'trend_direction'}
Identifiers (CONTEXT ONLY): {'client_id', 'content_id'}
Potential Features Count: 40


## 3. Success metric

*One metric you can defend. What number means "good"?*

### Primary Metric: Precision@50

Precision@50 measures the proportion of declining pages among the 50 pages placed at the top of the model's review queue.

For Lane 2, this metric directly reflects the operational question:

> **If the content team can review 50 pages first, how many of those pages are actually declining?**

The starter dataset has a declining base rate of **54.2%**. This provides a simple reference point for interpreting the top-50 result.

A useful ML model should improve Precision@50 compared with simple baseline approaches when evaluated on the same held-out data.

ROC-AUC and other metrics may be used later as additional diagnostics, but **Precision@50 is the primary success metric for this lane because the business output is a prioritized top-50 review queue.**

In [4]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

def precision_at_k(df, score_col, k=50):
    top_k = df.sort_values(
        by=score_col,
        ascending=False
    ).head(k)

    return top_k["is_declining_label"].mean()

base_rate = df["is_declining_label"].mean()

print("=== BASE RATE ===")
print(f"Overall declining rate: {base_rate:.1%}")

=== BASE RATE ===
Overall declining rate: 54.2%


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**Unit of Analysis:** One row = **one unique content page (`content_id`) for a specific client (`client_id`)**, with its observed search and engagement metrics from the starter dataset.

**Dataset Dimensions:** 30,000 rows × 44 columns.

The dataframe below shows the page-level unit of analysis using example fields such as content type, search volume, average position, CTR, freshness, and the observed trend label.

In [ ]:
# Code check: Display slice of unit of analysis dataframe
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print(f"Dataframe Shape: {df.shape}")
print(f"Unique content_id count: {df['content_id'].nunique():,}")
print(f"Unique client_id count: {df['client_id'].nunique():,}")

sample_cols = ['content_id', 'client_id', 'content_type', 'search_volume', 'avg_position', 'ctr', 'days_since_last_update', 'trend_direction']
df[sample_cols].head()

Dataframe Shape: (30000, 44)
Unique content_id count: 30,000
Unique client_id count: 32


,content_id,client_id,content_type,search_volume,avg_position,ctr,days_since_last_update,trend_direction
0,content_304f48230142,client_f369cb89fc,keyword article,10.0,10.6,0.76,20,down
1,content_a1fb4e703a9e,client_4e07408562,keyword article,90.0,20.3,0.05,25,down
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,0.0,36.5,0.09,20,down
3,content_331d6c4de07b,client_19581e27de,keyword article,10.0,6.2,0.49,22,stable
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,0.0,44.0,0.13,14,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

### Why Simple Rules May Not Be Enough

1. **Static heuristics are limited:** The `181+` freshness group has a 47.1% declining rate, which is lower than the overall 54.2% declining base rate. This suggests that content age alone does not fully explain observed decline.

2. **Multiple signals show different patterns:** Declining rates vary across observable groups such as freshness and position tier. This suggests that a single if-statement based on one signal may not capture the full pattern.

3. **ML can combine multiple signals:** A supervised model can potentially combine multiple legitimate pre-decision signals into a unified priority score. These features must pass leakage and feature-quality checks, and the model must demonstrate value by improving the agreed evaluation metric over simple baselines.

In [7]:
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

df["is_declining_label"] = (
    df["trend_direction"] == "down"
).astype(int)

freshness_decline = (
    df.groupby("freshness_tier", observed=False)["is_declining_label"]
      .agg(["count", "mean"])
)


print("=== DECLINE RATE BY FRESHNESS TIER ===")
print(age_decline)

pos_decline = (
    df.groupby("position_tier", observed=False)["is_declining_label"]
      .agg(["count", "mean"])
)

print("\n=== DECLINE RATE BY POSITION TIER ===")
print(pos_decline)

=== DECLINE RATE BY FRESHNESS TIER ===
                count      mean
freshness_tier                 
0-30            20480  0.511377
181+              174  0.471264
31-90             175  0.588571
91-180           9171  0.611057

=== DECLINE RATE BY POSITION TIER ===
               count      mean
position_tier                 
deep            1319  0.344200
page_1         11814  0.569663
page_3_5        7242  0.561585
striking        7304  0.609529
top_3           2321  0.240844


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.